# PyTorch (GPU) training path
TensorFlow GPU conv kernels were failing on this machine (RTX 5080 / CC 12.0) with `CUDA_ERROR_INVALID_HANDLE` even in a fresh process when building `Conv2D` models.

This notebook is now a clean PyTorch pipeline that keeps the same parameters (`image_size=(150,150)`, `batch_size=128`, `epochs=25`) and trains an EfficientNet-B0 classifier on the dataset folders (`seg_train/`, `seg_test/`).

In [1]:
# Install PyTorch + torchvision (GPU)
import sys
import subprocess

def pip_install(*packages: str):
    cmd = [sys.executable, "-m", "pip", "install", "-U", *packages]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)

# If this fails due to network/index restrictions, tell me what pip prints and I will adapt.
pip_install("torch", "torchvision")

Running: c:\Users\HFGF\miniconda3\envs\tf\python.exe -m pip install -U torch torchvision


In [1]:
# PyTorch imports + device setup (CUDA required)
import os
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import EfficientNet_B0_Weights

print("torch:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is NOT available in this kernel. "
        "You likely need to restart the notebook kernel after installing CUDA PyTorch, "
        "or you installed CPU-only wheels."
    )

device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))

# Performance knobs (safe defaults)
torch.backends.cudnn.benchmark = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

torch: 2.8.0+cu126
torch.version.cuda: 12.6
CUDA available: True
GPU: NVIDIA GeForce RTX 2060 SUPER
Compute capability: (7, 5)


In [2]:
# Parameters (kept the same as your TensorFlow version)
image_size = (150, 150)
batch_size = 128
epochs = 25
num_classes = 6

class_names = ["buildings", "forest", "glacier", "mountain", "sea", "street"]

In [ ]:
# Download dataset via KaggleHub (recommended so seg_test exists)
USE_KAGGLEHUB = True  # set False if you already have seg_train/ and seg_test/ locally

if USE_KAGGLEHUB:
    try:
        import kagglehub
    except ImportError:
        import sys, subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "kagglehub"])
        import kagglehub

    path = kagglehub.dataset_download("puneet6060/intel-image-classification")
    print("Path to dataset files:", path)

In [4]:
# Dataset paths (reuse your KaggleHub path if present; otherwise use repo-local folders)
def pick_existing_dir(*candidates: str) -> str:
    for d in candidates:
        if d and os.path.isdir(d):
            return d
    raise FileNotFoundError(f"None of these directories exist: {candidates}")

def unwrap_nested_dir(dir_path: str) -> str:
    """Handle Intel dataset layouts like seg_train/seg_train/<class> and seg_pred/seg_pred/*.jpg."""
    if not dir_path or not os.path.isdir(dir_path):
        return dir_path
    base = os.path.basename(os.path.normpath(dir_path))
    nested = os.path.join(dir_path, base)
    return nested if os.path.isdir(nested) else dir_path

dataset_root = globals().get("path", None)
train_dir = pick_existing_dir(
    os.path.join(dataset_root, "seg_train") if dataset_root else None,
    "seg_train",
 )
val_dir = pick_existing_dir(
    os.path.join(dataset_root, "seg_test") if dataset_root else None,
    "seg_test",
 )

train_dir = unwrap_nested_dir(train_dir)
val_dir = unwrap_nested_dir(val_dir)

print("Train dir:", train_dir)
print("Val dir:", val_dir)

Train dir: seg_train
Val dir: seg_test


In [5]:
# Transforms + DataLoaders
weights = EfficientNet_B0_Weights.DEFAULT
mean = weights.transforms().mean
std = weights.transforms().std

train_transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
 ])

val_transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
 ])

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)

print("Train classes:", train_dataset.classes)
print("Val classes:", val_dataset.classes)

# Make sure class ordering matches your expected names (optional)
if train_dataset.classes != class_names:
    print("Note: class order differs from class_names; using folder order:", train_dataset.classes)

# On Windows, too many workers can spike CPU + add overhead; start modest and scale up if GPU starves.
num_workers = min(4, os.cpu_count() or 0)
pin_memory = True  # since we hard-require CUDA
prefetch_factor = 2  # each worker preloads batches (ignored if num_workers=0)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=(num_workers > 0),
    prefetch_factor=prefetch_factor if num_workers > 0 else None,
 )
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=(num_workers > 0),
    prefetch_factor=prefetch_factor if num_workers > 0 else None,
 )

print("num_workers:", num_workers, "pin_memory:", pin_memory, "prefetch_factor:", prefetch_factor)
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

Train classes: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Val classes: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
num_workers: 4 pin_memory: True prefetch_factor: 2
Train batches: 110
Val batches: 24


In [6]:
# Model (EfficientNet-B0)
model = models.efficientnet_b0(weights=weights)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)
model = model.to(device)

# Hard checks: make sure the model is actually on GPU
assert next(model.parameters()).is_cuda, "Model parameters are not on CUDA"
print("Model device:", next(model.parameters()).device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

use_amp = True  # CUDA required, so AMP can be on
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

print(model.__class__.__name__)
print("AMP enabled:", use_amp)

Model device: cuda:0
EfficientNet
AMP enabled: True


C:\Users\HFGF\AppData\Local\Temp\ipykernel_13396\1575798397.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


In [9]:
# Quick GPU sanity check (one forward pass)
x = torch.randn(32, 3, image_size[0], image_size[1], device=device)
with torch.inference_mode(), torch.cuda.amp.autocast(enabled=use_amp):
    y = model(x)
torch.cuda.synchronize()
print("Sanity forward OK. Output shape:", tuple(y.shape))
print("GPU mem allocated (MB):", int(torch.cuda.memory_allocated() / 1024**2))

C:\Users\HFGF\AppData\Local\Temp\ipykernel_15304\888263783.py:3: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.inference_mode(), torch.cuda.amp.autocast(enabled=use_amp):


C:\Users\HFGF\AppData\Local\Temp\ipykernel_15304\888263783.py:3: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.inference_mode(), torch.cuda.amp.autocast(enabled=use_amp):


Sanity forward OK. Output shape: (32, 6)
GPU mem allocated (MB): 32


In [ ]:
# Training loop
from time import time

def accuracy(logits: torch.Tensor, targets: torch.Tensor) -> float:
    preds = torch.argmax(logits, dim=1)
    return (preds == targets).float().mean().item()

def run_epoch(loader: DataLoader, train: bool):
    model.train(train)
    total_loss = 0.0
    total_acc = 0.0
    total_samples = 0
    start = time()

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, targets)

        batch_size_local = targets.size(0)
        total_loss += loss.item() * batch_size_local
        total_acc += accuracy(logits.detach(), targets) * batch_size_local
        total_samples += batch_size_local

        if train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

    elapsed = time() - start
    return total_loss / max(1, total_samples), total_acc / max(1, total_samples), elapsed

os.makedirs("torch", exist_ok=True)

for epoch in range(1, epochs + 1):
    train_loss, train_acc, train_time = run_epoch(train_loader, train=True)
    val_loss, val_acc, val_time = run_epoch(val_loader, train=False)

    print(
        f"Epoch {epoch:02d}/{epochs} | "
        f"train loss {train_loss:.4f} acc {train_acc:.4f} ({train_time:.1f}s) | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} ({val_time:.1f}s)"
    )

    ckpt_path = f"torch/epoch_{epoch:02d}.pt"
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "class_names": train_dataset.classes,
            "image_size": image_size,
        },
        ckpt_path,
    )

C:\Users\HFGF\AppData\Local\Temp\ipykernel_8668\3006277935.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 01/25 | train loss 0.4186 acc 0.8664 (66.0s) | val loss 0.2056 acc 0.9287 (14.4s)
Epoch 02/25 | train loss 0.1711 acc 0.9404 (13.5s) | val loss 0.1914 acc 0.9283 (1.1s)
Epoch 03/25 | train loss 0.1286 acc 0.9556 (13.4s) | val loss 0.2057 acc 0.9270 (1.0s)
Epoch 04/25 | train loss 0.0980 acc 0.9635 (13.4s) | val loss 0.1871 acc 0.9390 (1.1s)
Epoch 05/25 | train loss 0.0699 acc 0.9751 (13.6s) | val loss 0.2207 acc 0.9353 (1.1s)
Epoch 06/25 | train loss 0.0615 acc 0.9781 (13.7s) | val loss 0.2303 acc 0.9330 (1.0s)
Epoch 07/25 | train loss 0.0454 acc 0.9838 (13.8s) | val loss 0.2518 acc 0.9347 (1.1s)
Epoch 08/25 | train loss 0.0456 acc 0.9835 (13.7s) | val loss 0.2599 acc 0.9373 (1.0s)
Epoch 09/25 | train loss 0.0425 acc 0.9853 (13.7s) | val loss 0.2610 acc 0.9367 (1.1s)
Epoch 10/25 | train loss 0.0309 acc 0.9891 (13.8s) | val loss 0.2793 acc 0.9380 (1.1s)
Epoch 11/25 | train loss 0.0320 acc 0.9885 (13.9s) | val loss 0.2465 acc 0.9383 (1.0s)
Epoch 12/25 | train loss 0.0286 acc 0.9897

In [10]:
# Verify on 100 random images from seg_pred/ (unlabeled)
import random
from pathlib import Path
from PIL import Image
import torch
import matplotlib.pyplot as plt

images_to_test = 3000

# Assumes earlier cells have defined: model, device, val_transform, class_names
need = ["model", "device", "val_transform", "class_names"]
missing = [n for n in need if n not in globals()]
if missing:
    raise RuntimeError(f"Missing {missing}. Run the setup/training cells first.")

ckpts = sorted(Path("torch").glob("epoch_*.pt"))
if ckpts:
    ckpt = torch.load(ckpts[-1], map_location="cpu")
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    model.to(device)
    print("Loaded checkpoint:", ckpts[-1])

dataset_root = globals().get("path", None)
seg_pred = Path(dataset_root) / "seg_pred" if dataset_root else Path("seg_pred")
if (seg_pred / "seg_pred").is_dir():
    seg_pred = seg_pred / "seg_pred"
if not seg_pred.is_dir():
    raise FileNotFoundError("seg_pred/ not found. Download the dataset first.")
print("seg_pred dir:", seg_pred.resolve())

img_paths = []
for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"):
    img_paths.extend(seg_pred.glob(ext))
if not img_paths:
    raise FileNotFoundError(f"No images found in {seg_pred}")
k = min(images_to_test, len(img_paths))
sample_paths = random.Random(42).sample(img_paths, k)
print(f"Sampling {k} images (total available: {len(img_paths)})")

model.eval()
batch = torch.stack([val_transform(Image.open(p).convert("RGB")) for p in sample_paths]).to(device)
with torch.inference_mode():
    probs = torch.softmax(model(batch), dim=1)
    conf, pred = probs.max(dim=1)
conf = conf.cpu().tolist()
pred = pred.cpu().tolist()

counts = {name: 0 for name in class_names}
for i in pred:
    counts[class_names[i]] += 1
print("\nPredicted class counts:")
for name, n in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])):
    print(f"  {name:10s} {n}")
print("Mean confidence:", sum(conf) / len(conf))

Loaded checkpoint: torch\epoch_25.pt
seg_pred dir: C:\Users\HFGF\Desktop\Programmering\image-recognition\seg_pred
Sampling 3000 images (total available: 7301)

Predicted class counts:
  street     534
  glacier    532
  mountain   519
  sea        515
  forest     465
  buildings  435
Mean confidence: 0.9799838213423888


In [ ]:
# Export a TRAINED checkpoint (.pt) to ONNX for the Next.js app
from pathlib import Path
import json
import torch
from torch import nn
from torchvision import models
from torchvision.models import EfficientNet_B0_Weights

# Paths
CKPT_DIR = Path("torch")  # where the training loop saved epoch_XX.pt
OUT_ONNX = Path("public/model/model.onnx")
OUT_LABELS = Path("public/model/labels.json")

# Pick the latest checkpoint
ckpts = sorted(CKPT_DIR.glob("epoch_*.pt"))
if not ckpts:
    raise FileNotFoundError(f"No checkpoints found in {CKPT_DIR.resolve()}. Train first so torch/epoch_XX.pt exists.")
ckpt_path = ckpts[-1]
print("Using checkpoint:", ckpt_path)

ckpt = torch.load(ckpt_path, map_location="cpu")
class_names = ckpt.get("class_names")
if not isinstance(class_names, list) or not all(isinstance(x, str) for x in class_names):
    raise ValueError("Checkpoint does not contain a valid class_names list")

num_classes = len(class_names)
image_size = tuple(ckpt.get("image_size", (150, 150)))
if image_size != (150, 150):
    print("Note: checkpoint image_size is", image_size, "but the web app assumes 150x150.")

# Recreate the exact model head and load weights
weights = EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)
model.load_state_dict(ckpt["model_state_dict"], strict=True)
model.eval()

# Export (NCHW input: [1,3,150,150])
OUT_ONNX.parent.mkdir(parents=True, exist_ok=True)
dummy = torch.randn(1, 3, image_size[0], image_size[1], dtype=torch.float32)
torch.onnx.export(
    model,
    dummy,
    str(OUT_ONNX),
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    do_constant_folding=True,
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
)
print("Wrote:", OUT_ONNX.resolve())

# Write labels so the API can map indices -> names
OUT_LABELS.write_text(json.dumps(class_names), encoding="utf-8")
print("Wrote:", OUT_LABELS.resolve())
print("Labels:", class_names)

C:\Users\HFGF\AppData\Local\Temp\ipykernel_13396\2669833276.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(m, torch.randn(1, 3, image_size[0], image_size[1]), "model.onnx", opset_version=17)


Wrote model.onnx
